# Template Example Notebook

This is a template notebook. The first heading should be the title of what notebook is about. For example, if it is a project on neo4j tutorial the heading should be `Project Title`.

- Add description of what the notebook does.
- Point to references, e.g. (neo4j.example.md)
- Add citations.
- Keep the notebook flow clear.
- Comments should be imperative and have a period at the end.
- Your code should be well commented.

The name of this notebook should in the following format:
- if the notebook is exploring `pycaret API`, then it is `pycaret.example.ipynb`

Follow the reference to write notebooks in a clear manner: https://github.com/causify-ai/helpers/blob/master/docs/coding/all.jupyter_notebook.how_to_guide.md

In [39]:
# %load_ext autoreload
# %autoreload 2
# %matplotlib inline

In [40]:
# import logging
# # Import libraries in this section.
# # Avoid imports like import *, from ... import ..., from ... import *, etc.

# import helpers.hdbg as hdbg
# import helpers.hnotebook as hnotebo

In [41]:
# hdbg.init_logger(verbosity=logging.INFO)

# _LOG = logging.getLogger(__name__)

# hnotebo.config_notebook()

## Make the notebook flow clear
Each notebook needs to follow a clear and logical flow, e.g:
- Load data
- Compute stats
- Clean data
- Compute stats
- Do analysis
- Show results




#############################################################################
Template
#############################################################################

In [42]:
# class Template:
#     """
#     Brief imperative description of what the class does in one line, if needed.
#     """

#     def __init__(self):
#         pass

#     def method1(self, arg1: int) -> None:
#         """
#         Brief imperative description of what the method does in one line.

#         You can elaborate more in the method docstring in this section, for e.g. explaining
#         the formula/algorithm. Every method/function should have a docstring, typehints and include the
#         parameters and return as follows:

#         :param arg1: description of arg1
#         :return: description of return
#         """
#         # Code bloks go here.
#         # Make sure to include comments to explain what the code is doing.
#         # No empty lines between code blocks.
#         pass


# def template_function(arg1: int) -> None:
#     """
#     Brief imperative description of what the function does in one line.

#     You can elaborate more in the function docstring in this section, for e.g. explaining
#     the formula/algorithm. Every function should have a docstring, typehints and include the
#     parameters and return as follows:

#     :param arg1: description of arg1
#     :return: description of return
#     """
#     # Code bloks go here.
#     # Make sure to include comments to explain what the code is doing.
#     # No empty lines between code blocks.
#     pass

## The flow should be highlighted using headings in markdown
```
# Level 1
## Level 2
### Level 3
```

# Description

This notebook implements a full time series forecasting pipeline using the
Darts library to predict S&P 500 index prices and identify which market
sectors will outperform in the near future.

The notebook covers data collection for S&P 500, 11 sector ETFs, and 22
macroeconomic dimensions including yield curve, VIX, CPI, Fed rate, oil,
gold, and dollar index. It performs data preprocessing with release-date-aware
forward fill, feature engineering including technical indicators, calendar
features, and event flags for FOMC meetings, CPI release dates, and US
holidays. It trains the full Darts model suite across baseline, statistical,
probabilistic, and machine learning model groups, applies SHAP based feature
selection, performs hyperparameter tuning, builds ensemble models, and
benchmarks results against Facebook Prophet and Statsmodels. The sector
rotation engine recommends which sectors to rotate into based entirely on
model predictions, historical correlations, and risk adjusted scores.

References:
- Darts documentation: https://unit8co.github.io/darts/
- FRED API documentation: https://fred.stlouisfed.org/docs/api/fred/
- yfinance documentation: https://pypi.org/project/yfinance/
- Prophet documentation: https://facebook.github.io/prophet/

# Imports

In [43]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import logging
import os
import warnings

import darts
import darts.dataprocessing.transformers
import darts.metrics
import darts.models
import darts.timeseries
import dotenv
import fredapi
import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import prophet
import seaborn as sns
import shap
import sklearn.preprocessing
import statsmodels.tsa.arima.model
import statsmodels.tsa.holtwinters
import tqdm
import utils
import yfinance as yf

warnings.filterwarnings("ignore")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
# Configure the logger to track notebook execution.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
_LOG = logging.getLogger(__name__)
# Log the environment setup confirmation and Darts version.
_LOG.info("Environment configured successfully.")
_LOG.info("Darts version: %s", darts.__version__)

2026-04-04 17:37:09,738 - INFO - Environment configured successfully.
2026-04-04 17:37:09,739 - INFO - Darts version: 0.43.0


# Configuration

In [45]:
# Define the date range for all data downloads.
START_DATE = "2018-01-01"
END_DATE = "2024-12-31"
# Define the S&P 500 ticker symbol.
SP500_TICKER = "^GSPC"
# Define all 11 sector ETF ticker symbols.
SECTOR_TICKERS = [
    "XLK",
    "XLV",
    "XLF",
    "XLE",
    "XLY",
    "XLP",
    "XLI",
    "XLU",
    "XLB",
    "XLRE",
    "XLC",
]
# Define sector names mapped to their ticker symbols.
SECTOR_NAMES = {
    "XLK": "Technology",
    "XLV": "Healthcare",
    "XLF": "Financials",
    "XLE": "Energy",
    "XLY": "Consumer Discretionary",
    "XLP": "Consumer Staples",
    "XLI": "Industrials",
    "XLU": "Utilities",
    "XLB": "Materials",
    "XLRE": "Real Estate",
    "XLC": "Communication Services",
}
# Define daily macro indicator ticker symbols from yfinance.
DAILY_MACRO_TICKERS = {
    "VIX": "^VIX",
    "TNX": "^TNX",
    "IRX": "^IRX",
    "OIL": "CL=F",
    "GOLD": "GC=F",
    "DXY": "DX-Y.NYB",
}
# Define monthly macro indicator codes from FRED API.
MONTHLY_MACRO_CODES = {
    "CPI": "CPIAUCSL",
    "CORE_CPI": "CPILFESL",
    "FED_RATE": "FEDFUNDS",
    "UNEMPLOYMENT": "UNRATE",
    "NFP": "PAYEMS",
    "RETAIL_SALES": "RSAFS",
    "INDUSTRIAL_PROD": "INDPRO",
    "PCE": "PCEPI",
    "PPI": "PPIACO",
}
# Define the FRED daily indicator code for breakeven inflation.
BREAKEVEN_INFLATION_CODE = "T10YIE"
# Define the data directory path for saving raw CSV files.
DATA_DIR = "data"
# Define the test set size in trading days.
TEST_SIZE = 60
# Define the validation set size in trading days.
VAL_SIZE = 60
# Define the forecast horizon in trading days.
FORECAST_HORIZON = 30
# Create the data directory if it does not exist.
os.makedirs(DATA_DIR, exist_ok=True)
_LOG.info("Configuration loaded successfully.")
_LOG.info("Date range: %s to %s", START_DATE, END_DATE)
_LOG.info("Sectors: %s", list(SECTOR_NAMES.values()))

2026-04-04 17:37:09,757 - INFO - Configuration loaded successfully.
2026-04-04 17:37:09,757 - INFO - Date range: 2018-01-01 to 2024-12-31
2026-04-04 17:37:09,757 - INFO - Sectors: ['Technology', 'Healthcare', 'Financials', 'Energy', 'Consumer Discretionary', 'Consumer Staples', 'Industrials', 'Utilities', 'Materials', 'Real Estate', 'Communication Services']


# Data collection

This section downloads all required data from Yahoo Finance and the FRED
API and saves it as raw CSV files in the `data` directory. The data
includes S&P 500 historical prices, 11 sector ETF prices, 6 daily
macroeconomic indicators, and 10 monthly macroeconomic indicators covering
the period from 2018 to 2024.

In [46]:
# Load environment variables from the .env file.
dotenv.load_dotenv()
# Read the FRED API key from the environment variables.
FRED_API_KEY = os.environ.get("FRED_API_KEY")
# Verify the FRED API key was loaded successfully.
if FRED_API_KEY is None:
    raise ValueError("FRED_API_KEY not found in .env file.")
_LOG.info("FRED API key loaded successfully.")

2026-04-04 17:37:09,773 - INFO - FRED API key loaded successfully.


In [47]:
# Load S&P 500 data from CSV if available otherwise download fresh.
if os.path.exists(os.path.join(DATA_DIR, "sp500_raw.csv")):
    sp500 = utils.load_data("sp500_raw.csv", DATA_DIR)
else:
    sp500 = utils.download_sp500(SP500_TICKER, START_DATE, END_DATE)
    utils.save_data(sp500, "sp500_raw.csv", DATA_DIR)
sp500.head(3)

2026-04-04 17:37:09,789 - INFO - Downloading S&P 500 data from 2018-01-01 to 2024-12-31.
2026-04-04 17:37:09,825 - INFO - Downloaded 1760 rows of S&P 500 data.
2026-04-04 17:37:09,832 - INFO - Saved 1760 rows to data/sp500_raw.csv.


Price,Close,High,Low,Open,Volume
Date,,,,,
2018-01-02,2695.810059,2695.889893,2682.360107,2683.729980,3397430000
2018-01-03,2713.060059,2714.370117,2697.770020,2697.850098,3544030000
2018-01-04,2723.989990,2729.290039,2719.070068,2719.310059,3697340000


In [48]:
# Load sector ETF data from CSV if available otherwise download fresh.
if os.path.exists(os.path.join(DATA_DIR, "sectors_raw.csv")):
    sectors = utils.load_data("sectors_raw.csv", DATA_DIR)
else:
    sectors = utils.download_sectors(SECTOR_TICKERS, START_DATE, END_DATE)
    utils.save_data(sectors, "sectors_raw.csv", DATA_DIR)
sectors.head(3)

2026-04-04 17:37:09,850 - INFO - Downloading 11 sector ETFs.
2026-04-04 17:37:09,997 - INFO - Downloaded 1760 rows for 11 sectors.
2026-04-04 17:37:10,006 - INFO - Saved 1760 rows to data/sectors_raw.csv.


,XLK,XLV,XLF,XLE,XLY,XLP,XLI,XLU,XLB,XLRE,XLC
Date,,,,,,,,,,,
2018-01-02,29.872921,72.723335,23.884142,25.747267,46.275246,45.314571,66.254837,20.132584,25.995680,24.837584,NaN
2018-01-03,30.122097,73.419182,24.012455,26.132853,46.487698,45.298527,66.611710,19.974422,26.177759,24.845165,NaN
2018-01-04,30.274364,73.523575,24.234877,26.290606,46.640129,45.426777,67.099136,19.808550,26.406427,24.420458,NaN


In [49]:
# Load daily macro data from CSV if available otherwise download fresh.
if os.path.exists(os.path.join(DATA_DIR, "macro_daily_raw.csv")):
    macro_daily = utils.load_data("macro_daily_raw.csv", DATA_DIR)
else:
    macro_daily = utils.download_daily_macro(
        DAILY_MACRO_TICKERS, START_DATE, END_DATE
    )
    utils.save_data(macro_daily, "macro_daily_raw.csv", DATA_DIR)
macro_daily.head(3)

2026-04-04 17:37:10,023 - INFO - Downloading 6 daily macro indicators.
2026-04-04 17:37:10,107 - INFO - Downloaded 1760 rows for 6 daily macro indicators.
2026-04-04 17:37:10,113 - INFO - Saved 1760 rows to data/macro_daily_raw.csv.


,VIX,TNX,IRX,OIL,GOLD,DXY
Date,,,,,,
2018-01-02,9.77,2.465,1.378,60.369999,1313.699951,91.849998
2018-01-03,9.15,2.447,1.370,61.630001,1316.199951,92.160004
2018-01-04,9.22,2.453,1.370,62.009998,1319.400024,91.849998


In [ ]:
# Load monthly macro data from CSV if available otherwise download fresh.
if os.path.exists(os.path.join(DATA_DIR, "macro_monthly_raw.csv")):
    macro_monthly = utils.load_data("macro_monthly_raw.csv", DATA_DIR)
else:
    macro_monthly = utils.download_monthly_macro(
        MONTHLY_MACRO_CODES,
        BREAKEVEN_INFLATION_CODE,
        START_DATE,
        END_DATE,
        FRED_API_KEY,
    )
    utils.save_data(macro_monthly, "macro_monthly_raw.csv", DATA_DIR)
macro_monthly.head(3)

2026-04-04 17:37:10,128 - INFO - Downloading 9 monthly macro indicators from FRED.

# Data preprocessing and EDA

This section cleans and aligns all downloaded datasets, applies
release-date-aware forward fill to monthly macro indicators, adds
binary release flags to distinguish actual data release days from
forward filled values, and performs exploratory data analysis to
understand the structure and patterns in the data before modeling.

## Align datasets to common date index

In [ ]:
# Use S&P 500 date index as the master index for all datasets.
master_index = sp500.index
# Reindex daily datasets directly — no monthly values to preserve.
sectors = sectors.reindex(master_index)
macro_daily = macro_daily.reindex(master_index)
# Preserve monthly values before reindexing to avoid losing data
# that falls on non trading days like weekends and holidays.
macro_monthly = utils.preserve_month_start_values(
    macro_monthly, master_index
)
# Verify all datasets have the same shape after alignment.
_LOG.info("S&P 500 shape: %s", sp500.shape)
_LOG.info("Sectors shape: %s", sectors.shape)
_LOG.info("Macro daily shape: %s", macro_daily.shape)
_LOG.info("Macro monthly shape: %s", macro_monthly.shape)

## Apply release-date-aware forward fill to monthly indicators

In [ ]:
# Apply release-date-aware forward fill to monthly macro indicators.
macro_monthly = utils.apply_release_aware_forward_fill(
    macro_monthly,
    FRED_API_KEY,
    MONTHLY_MACRO_CODES,
)
# Preview the result to verify forward fill worked correctly.
macro_monthly.head(20)

In [ ]:
# Check when the first non-NaN values appear for each indicator.
first_valid = macro_monthly.apply(lambda col: col.first_valid_index())
_LOG.info("First valid date per indicator:\n%s", first_valid.to_string())

In [ ]:
# Verify the forward fill and release flags worked correctly.
_LOG.info("Macro monthly shape: %s", macro_monthly.shape)
_LOG.info("Total NaN values remaining: %d", macro_monthly.isnull().sum().sum())
_LOG.info("Release flag columns: %s", 
    [col for col in macro_monthly.columns if col.endswith("_released")]
)
# Show a sample around a known release date to verify flags.
macro_monthly.loc["2018-01-01":"2018-02-05", ["CPI", "CPI_released"]].head(10)

In [ ]:
# Verify CPI release flag shows 1 on actual release days.
cpi_releases = macro_monthly[macro_monthly["CPI_released"] == 1]["CPI"]
_LOG.info(
    "Number of actual CPI release days: %d", len(cpi_releases)
)
_LOG.info(
    "First few CPI release dates:\n%s", cpi_releases.head(5).to_string()
)

In [ ]:
# Check fredapi version and available methods.
import fredapi
_LOG.info("fredapi version: %s", fredapi.__version__)
fred_temp = fredapi.Fred(api_key=FRED_API_KEY)
fred_methods = [m for m in dir(fred_temp) if not m.startswith("_")]
_LOG.info("Available methods: %s", fred_methods)

In [ ]:
# Verify CPI release flag shows 1 on true release days.
cpi_releases = macro_monthly[macro_monthly["CPI_released"] == 1]["CPI"]
_LOG.info("Number of actual CPI release days: %d", len(cpi_releases))
_LOG.info(
    "First few CPI release dates:\n%s", cpi_releases.head(5).to_string()
)
# Verify total NaN values remaining.
_LOG.info(
    "Total NaN values remaining: %d",
    macro_monthly.isnull().sum().sum()
)

In [ ]:
# Identify which columns still have NaN values and how many.
nan_counts = macro_monthly.isnull().sum()
nan_counts = nan_counts[nan_counts > 0]
_LOG.info("Columns with remaining NaN values:\n%s", nan_counts.to_string())

In [ ]:
# Find the first date where all monthly macro indicators have values.
first_complete_date = macro_monthly.dropna().index[0]
_LOG.info(
    "First date with complete macro data: %s", first_complete_date
)
# Count how many rows we lose by starting from this date.
rows_before = len(macro_monthly)
rows_after = len(macro_monthly.loc[first_complete_date:])
rows_dropped = rows_before - rows_after
_LOG.info(
    "Rows dropped: %d (%.2f%% of total)",
    rows_dropped,
    rows_dropped / rows_before * 100,
)

## Trim datasets to first complete macro data date

In [ ]:
# Trim all datasets to start from the first date where all macro
# indicators have complete values to eliminate initial NaN period.
sp500 = sp500.loc[first_complete_date:]
sectors = sectors.loc[first_complete_date:]
macro_daily = macro_daily.loc[first_complete_date:]
macro_monthly = macro_monthly.loc[first_complete_date:]
# Show percentage of rows dropped across all datasets.
_LOG.info(
    "Rows dropped: %d (%.2f%% of total)",
    rows_dropped,
    rows_dropped / rows_before * 100,
)
# Verify all datasets have the same shape after trimming.
_LOG.info("S&P 500 shape after trim: %s", sp500.shape)
_LOG.info("Sectors shape after trim: %s", sectors.shape)
_LOG.info("Macro daily shape after trim: %s", macro_daily.shape)
_LOG.info("Macro monthly shape after trim: %s", macro_monthly.shape)
# Verify no NaN values remain in monthly macro data.
_LOG.info(
    "NaN values remaining in macro monthly: %d",
    macro_monthly.isnull().sum().sum()
)

## Reconstruct XLC pre-launch history

XLC (Communication Services ETF) launched on June 19 2018. To preserve
the full analysis period from January 2018, pre-launch XLC values are
reconstructed using historical prices of the top 12 XLC constituent
companies weighted by market capitalization at each point in time.

The reconstruction is validated against actual XLC prices for the
overlap period from June 2018 onwards. If the correlation exceeds
0.90 the reconstruction replaces NaN values in the sectors DataFrame.
If correlation falls below 0.90 the dataset is trimmed to June 2018
as a fallback — ensuring we never use an unreliable reconstruction.

References:
- XLC constituent data: https://www.ssga.com/us/en/intermediary/etfs/funds/the-communication-services-select-sector-spdr-fund-xlc

In [ ]:
# Define XLC launch date as the first valid date in the sectors data.
xlc_first_date = pd.Timestamp("2018-06-19")
# Reconstruct XLC pre-launch history using verified constituent weights
# from the official SPDR ETF filing on June 19 2018.
sectors = utils.reconstruct_xlc(
    sectors,
    START_DATE,
    str(xlc_first_date.date()),
    correlation_threshold=0.90,
)
# Verify XLC NaN values are resolved after reconstruction.
xlc_nan_remaining = sectors["XLC"].isnull().sum()
_LOG.info(
    "XLC NaN values remaining after reconstruction: %d",
    xlc_nan_remaining,
)
# Show percentage of rows that were reconstructed.
_LOG.info(
    "XLC rows reconstructed: %d (%.2f%% of total)",
    98 - xlc_nan_remaining,
    (98 - xlc_nan_remaining) / len(sectors) * 100,
)
# Preview XLC values around the reconstruction period.
sectors[["XLC"]].loc[
    sectors.index[sectors.index < pd.Timestamp(xlc_first_date)]
].tail(10)

In [ ]:
# Verify the full sectors DataFrame has zero NaN values.
_LOG.info(
    "Total NaN values in sectors: %d",
    sectors.isnull().sum().sum()
)
# Verify all sectors start from the same date.
_LOG.info("Sectors date range: %s to %s",
    sectors.index[0].date(),
    sectors.index[-1].date()
)
# Preview sectors DataFrame.
sectors.head(10)

## Check and handle missing values in daily macro indicators

In [ ]:
# Check missing values across all datasets before proceeding.
_LOG.info("Checking missing values across all datasets.")
# Count NaN values per column for each dataset.
sp500_nan = sp500.isnull().sum()
sectors_nan = sectors.isnull().sum()
macro_daily_nan = macro_daily.isnull().sum()
macro_monthly_nan = macro_monthly.isnull().sum()
# Log results for each dataset.
_LOG.info(
    "S&P 500 missing values:\n%s", sp500_nan[sp500_nan > 0].to_string()
)
_LOG.info(
    "Sectors missing values:\n%s",
    sectors_nan[sectors_nan > 0].to_string()
)
_LOG.info(
    "Macro daily missing values:\n%s",
    macro_daily_nan[macro_daily_nan > 0].to_string()
)
_LOG.info(
    "Macro monthly missing values:\n%s",
    macro_monthly_nan[macro_monthly_nan > 0].to_string()
)
# Log total missing values across all datasets.
total_nan = (
    sp500_nan.sum()
    + sectors_nan.sum()
    + macro_daily_nan.sum()
    + macro_monthly_nan.sum()
)
_LOG.info("Total missing values across all datasets: %d", total_nan)

In [ ]:
# Forward fill any remaining missing values in daily macro indicators.
# Single missing values in daily data are caused by occasional gaps
# in yfinance data — forward fill is the correct approach here.
macro_daily = macro_daily.ffill()
# Verify no missing values remain after forward fill.
_LOG.info(
    "Missing values after forward fill: %d",
    macro_daily.isnull().sum().sum(),
)

In [ ]:
# Find the exact location of the remaining missing value.
missing_location = macro_daily[macro_daily["GOLD"].isnull()]
_LOG.info(
    "Missing GOLD value location:\n%s",
    missing_location[["GOLD"]].to_string()
)

In [ ]:
# Forward fill any remaining missing values in daily macro indicators.
# Single missing values in daily data are caused by occasional gaps
# in yfinance data — forward fill is the correct approach here.
macro_daily = macro_daily.ffill()
# Backward fill any remaining missing values at the start of the
# dataset where forward fill cannot work due to no prior values.
macro_daily = macro_daily.bfill()
# Verify no missing values remain after both fill operations.
_LOG.info(
    "Missing values after forward and backward fill: %d",
    macro_daily.isnull().sum().sum(),
)

In [ ]:
# Save all processed datasets to CSV so preprocessing never needs
# to be rerun on every kernel restart.
utils.save_data(sp500, "sp500_processed.csv", DATA_DIR)
utils.save_data(sectors, "sectors_processed.csv", DATA_DIR)
utils.save_data(macro_daily, "macro_daily_processed.csv", DATA_DIR)
utils.save_data(macro_monthly, "macro_monthly_processed.csv", DATA_DIR)
_LOG.info("All processed datasets saved successfully.")

# Exploratory data analysis

This section visualizes the structure and patterns in our cleaned
datasets before modeling. The analysis covers S&P 500 price history
and returns, sector performance comparison, macroeconomic indicator
trends, and correlations between macro signals and market returns.
These insights directly inform our feature engineering and model
selection decisions.

## S&P 500 price history and daily returns

In [ ]:
# Plot S&P 500 price history and daily returns.
fig = utils.plot_sp500_history(sp500)
plt.show()

## Sector performance comparison

In [ ]:
# Plot normalized cumulative performance of all 11 sector ETFs.
fig = utils.plot_sector_performance(sectors, SECTOR_NAMES)
plt.show()

## Macroeconomic indicator trends

In [ ]:
# Plot daily macroeconomic indicator trends with key market events highlighted.
fig = utils.plot_macro_trends(macro_daily, macro_monthly)
plt.show()

## Correlation between macro indicators and S&P 500 returns

In [ ]:
# Plot correlation between macro indicators and S&P 500 daily returns.
fig = utils.plot_macro_correlation(sp500, macro_daily, macro_monthly)
plt.show()

In [ ]:
# Plot rolling correlations between key macro indicators and S&P 500
# to reveal how relationships change across different market regimes.
fig = utils.plot_rolling_correlations(sp500, macro_daily, macro_monthly)
plt.show()